# Session 3: what happens in the seconds after a power station trips
**A. Real-world problem.** A large generator suddenly disconnects. Demand has not changed, so
the grid is short of power. The rotating machines give up some of their stored kinetic energy,
the frequency falls, the governors open the steam or water valves, and the frequency recovers
to a value slightly below 50 Hz. With wind and solar (little rotating mass) the fall is faster.

**Plan.** (1) write the swing equation in per unit, (2) solve it with our own RK4, (3) check the
start slope and the final value against hand formulas, (4) compare with and without governor,
(5) vary inertia and droop, (6) a three-bus DC power flow, (7) a failure case and a mini-project.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

F0 = 50.0   # nominal frequency (Hz)

## Step 1: the swing equation in per unit
Let dw = (f - f0) / f0 be the per-unit frequency deviation. For one aggregate generator
with inertia constant H (seconds of stored energy at rated power), damping D (the load falls
a little as frequency falls), mechanical power change Pm and a sudden loss dP (all per unit
on the generation base):
    2 H d(dw)/dt = Pm - dP - D dw          (same as 2H/f0 d(df)/dt = ...)
The governor with droop R and a combined turbine/governor lag Tg:
    Tg dPm/dt = -dw / R - Pm
An optional fast battery adds power Pb = clip(-dw / Rb, 0, B), reached with lag Tb:
    Tb dPb/dt = clip(-dw / Rb, 0, B) - Pb;     it enters the swing equation as + Pb.
Hand formulas to test the solver:
    initial rate of change   RoCoF = -f0 dP / (2 H)       (the governor has not moved yet)
    final deviation          df = -f0 dP / (1/R + D)      (Pm settles at -dw/R)

In [ ]:
def simulate_frequency(h_s=5.0, dp=0.05, droop=0.05, damp=1.0, tg=5.0, governor=True,
                       batt_pu=0.0, rb=0.02, tb=0.5, t_end=100.0, dt=0.02):
    """RK4 solution of the toy frequency model after a sudden loss dp (per unit).

    Returns (time in s, frequency in Hz, governor power Pm in pu, battery power Pb in pu).
    """
    def f(y):
        dw, pm, pb = y
        gov_cmd = -dw / droop if governor else 0.0
        bat_cmd = min(max(-dw / rb, 0.0), batt_pu)
        return np.array([(pm + pb - dp - damp * dw) / (2.0 * h_s),
                         (gov_cmd - pm) / tg,
                         (bat_cmd - pb) / tb])

    n = int(round(t_end / dt))
    y = np.zeros(3)
    out = np.zeros((n + 1, 3))
    for i in range(n):
        k1 = f(y)
        k2 = f(y + dt / 2 * k1)
        k3 = f(y + dt / 2 * k2)
        k4 = f(y + dt * k3)
        y = y + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        out[i + 1] = y
    t = np.arange(n + 1) * dt
    return t, F0 * (1.0 + out[:, 0]), out[:, 1], out[:, 2]


def rocof_analytic(dp, h_s):
    """Initial rate of change of frequency (Hz/s): -f0 dP / (2 H)."""
    return -F0 * dp / (2.0 * h_s)


def steady_deviation(dp, droop, damp=1.0):
    """Final frequency change (Hz) with governor droop: -f0 dP / (1/R + D)."""
    return -F0 * dp / (1.0 / droop + damp)


def nadir(freq):
    """Lowest frequency (Hz) reached."""
    return float(np.min(freq))

## Step 2: compare, then vary inertia and droop
Without a governor the frequency only stops falling because the load itself relaxes (damping D):
the final value is -f0 dP / D. With the governor the fall is stopped much earlier.

In [ ]:
def nadir_for(h_s, dp=0.05, droop=0.05, **kw):
    """Convenience: nadir (Hz) of one run."""
    return nadir(simulate_frequency(h_s=h_s, dp=dp, droop=droop, **kw)[1])


def trips_under_frequency(freq, limit=49.0):
    """True if the frequency ever goes below the under-frequency trip limit (Hz)."""
    return bool(np.min(freq) < limit)

## Step 3: a three-bus DC power flow (stretch)
The "DC" approximation of AC load flow: lines are reactances x (per unit), voltages are 1 pu, angle
differences are small, and real power in a line from i to j is (theta_i - theta_j) / x.
For the whole network B theta = P, with B built from 1/x. Bus 1 is the slack (angle 0).

In [ ]:
LINES = [(0, 1, 0.10), (0, 2, 0.20), (1, 2, 0.25)]   # (from bus, to bus, reactance in pu)


def dc_power_flow(p_inject, lines=LINES):
    """Solve B theta = P with bus 0 as slack. p_inject: net power at each bus (pu); the slack takes the rest.

    Returns (angles in radians, list of line flows in pu, slack power).
    """
    n = len(p_inject)
    b = np.zeros((n, n))
    for i, j, x in lines:
        b[i, i] += 1.0 / x
        b[j, j] += 1.0 / x
        b[i, j] -= 1.0 / x
        b[j, i] -= 1.0 / x
    p = np.asarray(p_inject, dtype=float).copy()
    theta = np.zeros(n)
    theta[1:] = np.linalg.solve(b[1:, 1:], p[1:])
    flows = [(theta[i] - theta[j]) / x for i, j, x in lines]
    return theta, flows, float(b[0] @ theta)

## Step 4: see it, then the mini-project
Mini-project: the smallest fast battery that keeps the nadir at or above 49.5 Hz after an 8 percent
loss on a low-inertia grid (H = 3 s).

In [ ]:
def smallest_battery(dp, h_s, target=49.5, step=0.005, max_pu=0.5, **kw):
    """Smallest battery size (pu, multiples of step) that keeps the nadir >= target Hz, or None."""
    k = 0
    while k * step <= max_pu + 1e-12:
        if nadir_for(h_s, dp=dp, batt_pu=k * step, **kw) >= target:
            return round(k * step, 6)
        k += 1
    return None


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t, f_gov, pm, _ = simulate_frequency()
    _, f_free, _, _ = simulate_frequency(governor=False)
    print(f"5% loss, H = 5 s: RoCoF analytic {rocof_analytic(0.05, 5.0):.3f} Hz/s, "
          f"numeric {(f_gov[5] - f_gov[0]) / t[5]:.3f} Hz/s")
    print(f"with governor: nadir {nadir(f_gov):.3f} Hz at t = {t[np.argmin(f_gov)]:.1f} s, "
          f"final {f_gov[-1]:.3f} Hz (analytic {F0 + steady_deviation(0.05, 0.05):.3f} Hz)")
    print(f"without governor: nadir {nadir(f_free):.3f} Hz, final {f_free[-1]:.3f} Hz "
          f"(analytic {F0 + -F0 * 0.05 / 1.0:.3f} Hz)")

    for h in (2.0, 3.0, 5.0, 8.0):
        print(f"H = {h:.0f} s: nadir {nadir_for(h):.3f} Hz")
    for r in (0.03, 0.05, 0.08):
        print(f"droop {r * 100:.0f}%: nadir {nadir_for(5.0, droop=r):.3f} Hz, final {F0 + steady_deviation(0.05, r):.3f} Hz")
    f_low = simulate_frequency(h_s=2.0, dp=0.10)[1]
    f_high = simulate_frequency(h_s=5.0, dp=0.10)[1]
    print(f"failure case, 10% loss: H = 2 s nadir {nadir(f_low):.3f} Hz (trip below 49 Hz: {trips_under_frequency(f_low)}); "
          f"H = 5 s nadir {nadir(f_high):.3f} Hz (trip: {trips_under_frequency(f_high)})")

    th, fl, ps = dc_power_flow([0.0, 0.5, -1.0])
    print("DC flow angles (rad):", np.round(th, 4), " flows 1-2, 1-3, 2-3 (pu):", np.round(fl, 4), " slack:", round(ps, 4))

    best = smallest_battery(0.08, 3.0)
    print(f"mini-project: 8% loss, H = 3 s, no battery nadir {nadir_for(3.0, dp=0.08):.3f} Hz; "
          f"smallest battery {best} pu gives {nadir_for(3.0, dp=0.08, batt_pu=best):.3f} Hz")

    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(t, f_gov, label="with governor (droop 5%)")
    ax.plot(t, f_free, "--", label="without governor")
    ax.axhline(49.0, color="r", ls=":", label="49 Hz trip level (toy)")
    ax.set_xlim(0, 60)
    ax.set_xlabel("time after the loss (s)")
    ax.set_ylabel("frequency (Hz)")
    ax.set_title("5 percent generation loss, H = 5 s")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "frequency_response.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(7, 4.2))
    for h in (2.0, 3.0, 5.0, 8.0):
        tt, ff, _, _ = simulate_frequency(h_s=h, dp=0.10)
        ax2.plot(tt, ff, label=f"H = {h:.0f} s")
    ax2.axhline(49.0, color="r", ls=":", label="49 Hz trip level (toy)")
    ax2.set_xlim(0, 40)
    ax2.set_xlabel("time after the loss (s)")
    ax2.set_ylabel("frequency (Hz)")
    ax2.set_title("Less inertia: faster and deeper fall (10 percent loss)")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "inertia_effect.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Change the loss to 12 percent, the droop to 3 percent and the battery size, and watch the nadir.
Then answer the four questions in the README (section F) and run the checkpoint.

## Coding tasks
1. **Easy.** Run `nadir_for` for H = 4 s with a 6 percent loss and with droop of 3 percent and 8 percent. Predict which droop gives the higher nadir first.
2. **Stretch.** Change `LINES` so that the line from bus 1 to bus 3 has a reactance of 0.4 and print the new flows. Check the power balance yourself.
3. **Explain.** Explain why the nadir with a governor of droop 5 percent is the same order as the settled value for large H but is lower for small H.

## Assessment
- **A. Conceptual.** Why does a grid with a lot of solar and wind (little spinning mass) see a faster and deeper frequency fall after the same loss?
- **B. Numerical.** An 8 percent loss hits a grid with H = 4 s. Find the initial RoCoF. Then for a 5 percent loss with droop R = 0.04 and D = 1, find the settled frequency.
- **C. Graph and simulation interpretation.** In the inertia figure (10 percent loss), compare the initial slopes for H = 2 s and H = 5 s, and say which curve crosses 49 Hz.
- **D. Design and optimisation challenge.** Design: after an 8 percent loss on a grid with H = 3 s the nadir must stay at or above 49.5 Hz. The nadir with no battery is 49.266 Hz. Find the smallest fast battery (in steps of 0.005 pu), and discuss what else could achieve it.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    t, f, pm, _ = L.simulate_frequency()
    checks.append(("worked example: 5% loss, H = 5 s gives RoCoF -0.250 Hz/s by hand (-f0 dP / 2H)",
                   abs(L.rocof_analytic(0.05, 5.0) + 0.25) < 1e-12))
    slope = (f[5] - f[0]) / t[5]
    checks.append(("the RK4 start slope matches the analytic RoCoF within 1 percent", abs(slope - L.rocof_analytic(0.05, 5.0)) < 0.0025))
    checks.append(("worked example: final frequency with governor is 49.881 Hz (analytic -f0 dP / (1/R + D))",
                   abs(f[-1] - 49.881) < 0.002 and abs(f[-1] - (50.0 + L.steady_deviation(0.05, 0.05))) < 0.002))
    checks.append(("the governor power settles at the loss: Pm = 0.0476 pu (0.05 minus D dw)", abs(pm[-1] - 0.0476) < 0.0005))
    _, f0n, _, _ = L.simulate_frequency(governor=False)
    checks.append(("without governor the final frequency is 47.500 Hz (-f0 dP / D)", abs(f0n[-1] - 47.5) < 0.005))
    k = int(round(10.0 / 0.02))
    exact = 50.0 * (1.0 - 0.05 * (1.0 - np.exp(-1.0 * t[k] / (2 * 5.0))))
    checks.append(("without governor the RK4 curve matches the exact exponential at t = 10 s", abs(f0n[k] - exact) < 1e-6))
    ta, fa, _, _ = L.simulate_frequency(dt=0.01)
    checks.append(("halving the time step changes the nadir by under 0.0001 Hz (RK4 converged)", abs(L.nadir(fa) - L.nadir(f)) < 1e-4))
    checks.append(("the governor lifts the nadir from 47.500 Hz to 49.627 Hz at H = 5 s",
                   abs(L.nadir(f0n) - 47.5) < 0.005 and abs(L.nadir(f) - 49.627) < 0.001))
    hs = [2.0, 3.0, 5.0, 8.0]
    nad = [L.nadir_for(h) for h in hs]
    checks.append(("less inertia gives a deeper nadir: 49.461, 49.541, 49.627, 49.692 Hz for H = 2, 3, 5, 8 s",
                   all(np.diff(nad) > 0) and np.allclose(nad, [49.461, 49.541, 49.627, 49.692], atol=0.001)))
    checks.append(("RoCoF scales as 1/H: H = 2 s is 2.5 times steeper than H = 5 s",
                   abs(L.rocof_analytic(0.05, 2.0) / L.rocof_analytic(0.05, 5.0) - 2.5) < 1e-12))
    dn = [L.nadir_for(5.0, droop=r) for r in (0.03, 0.05, 0.08)]
    checks.append(("droop sensitivity: nadir 49.707, 49.627, 49.535 Hz for R = 3, 5, 8 percent",
                   np.allclose(dn, [49.707, 49.627, 49.535], atol=0.001) and all(np.diff(dn) < 0)))
    checks.append(("final values for R = 3 and 8 percent are 49.927 and 49.815 Hz",
                   abs(50.0 + L.steady_deviation(0.05, 0.03) - 49.927) < 0.001 and abs(50.0 + L.steady_deviation(0.05, 0.08) - 49.815) < 0.001))
    f2 = L.simulate_frequency(h_s=2.0, dp=0.10)[1]
    f5 = L.simulate_frequency(h_s=5.0, dp=0.10)[1]
    checks.append(("failure case: a 10% loss at H = 2 s dips to 48.921 Hz and crosses 49 Hz; at H = 5 s the nadir is 49.253 Hz and it does not",
                   L.trips_under_frequency(f2) and not L.trips_under_frequency(f5)
                   and abs(L.nadir(f2) - 48.921) < 0.001 and abs(L.nadir(f5) - 49.253) < 0.001))
    checks.append(("edge: no loss means the frequency stays at exactly 50 Hz", np.allclose(L.simulate_frequency(dp=0.0)[1], 50.0, atol=1e-12)))
    th, fl, ps = L.dc_power_flow([0.0, 0.5, -1.0])
    checks.append(("three-bus DC flow: angles 0, 0.0045, -0.1091 rad; flows -0.0455, 0.5455, 0.4545 pu",
                   np.allclose(th, [0.0, 0.0045, -0.1091], atol=1e-4) and np.allclose(fl, [-0.0455, 0.5455, 0.4545], atol=1e-4)))
    checks.append(("DC flow power balance: slack supplies 0.5 pu, the slack flows add to it, and bus 3 receives 1.0 pu",
                   abs(ps - 0.5) < 1e-12 and abs(fl[0] + fl[1] - 0.5) < 1e-12 and abs(fl[1] + fl[2] - 1.0) < 1e-12
                   and abs(-fl[0] + fl[2] - 0.5) < 1e-12))
    checks.append(("DC flow hand check: with the slack at 0, B theta = P reproduces the injections",
                   abs((th[1] - th[0]) / 0.10 + (th[1] - th[2]) / 0.25 - 0.5) < 1e-12))
    b0 = L.nadir_for(3.0, dp=0.08)
    best = L.smallest_battery(0.08, 3.0)
    below = L.nadir_for(3.0, dp=0.08, batt_pu=best - 0.005)
    checks.append(("mini-project: 8% loss at H = 3 s gives 49.266 Hz with no battery; smallest battery is 0.035 pu (49.531 Hz); 0.030 pu is not enough",
                   abs(b0 - 49.266) < 0.001 and abs(best - 0.035) < 1e-9 and below < 49.5
                   and abs(L.nadir_for(3.0, dp=0.08, batt_pu=best) - 49.531) < 0.001))
    checks.append(("Section F answers: 8% loss at H = 4 s has RoCoF -0.5 Hz/s; 10% loss has -1.25 Hz/s at H = 2 s and -0.5 Hz/s at H = 5 s",
                   abs(L.rocof_analytic(0.08, 4.0) + 0.5) < 1e-12 and abs(L.rocof_analytic(0.10, 2.0) + 1.25) < 1e-12 and abs(L.rocof_analytic(0.10, 5.0) + 0.5) < 1e-12))
    checks.append(("a 4 percent droop gives a final frequency of 49.904 Hz after a 5% loss", abs(50.0 + L.steady_deviation(0.05, 0.04) - 49.904) < 0.001))
    checks.append(("the swing equation uses 2H = 10 s for H = 5 s: with no damping, governor or battery the first-second fall is 0.25 Hz",
                   abs(L.simulate_frequency(governor=False, damp=0.0, t_end=1.0)[1][-1] - 49.75) < 0.002))
    checks.append(("roadmap baselines: 10% loss at H = 2 s with a 0.05 pu battery gives 49.329 Hz; a 10% loss settles at 49.762 Hz; a 20% loss at H = 2 s dips to 47.84 Hz; headroom 0.06 pu would settle at 48.0 Hz",
                   abs(L.nadir_for(2.0, dp=0.10, batt_pu=0.05) - 49.329) < 0.001 and abs(50.0 + L.steady_deviation(0.10, 0.05) - 49.762) < 0.001
                   and abs(L.nadir_for(2.0, dp=0.20) - 47.84) < 0.005 and abs(50.0 * (1.0 - (0.10 - 0.06) / 1.0) - 48.0) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
